# FC, FCD, phFCD per model

Adapted from `wholebrain_comparison_plots.ipynb`. Runs each model's forward once, caches the results to disk (so you can re-plot without re-simulating), and then draws clean FC / FCD / phFCD matrices per model with no ticks or labels.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'examples':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

plt.rcParams['figure.dpi'] = 120

In [ ]:
# Auto-create parent dirs for savefig (added for reproducible runs)
import matplotlib.figure as _mpl_figure
from pathlib import Path as _Path
_orig_savefig = _mpl_figure.Figure.savefig
def _patched_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, (str, _Path)):
        p = _Path(fname)
        if p.parent and not p.parent.exists():
            p.parent.mkdir(parents=True, exist_ok=True)
    return _orig_savefig(self, fname, *args, **kwargs)
_mpl_figure.Figure.savefig = _patched_savefig


In [ ]:
# Reproducibility seed (added for publication run)
import random as _random
import numpy as _np
import torch as _torch
_SEED = 0
_random.seed(_SEED)
_np.random.seed(_SEED)
_torch.manual_seed(_SEED)
if _torch.cuda.is_available():
    _torch.cuda.manual_seed_all(_SEED)


## Configuration

In [ ]:
DATASET_TYPE = os.environ.get('DATASET_TYPE', 'ts_young')
DATA_PATH = 'data/ts_young/ts_young_TR0.72.mat'
CHECKPOINT_DIR = 'checkpoints'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

N_TIMEPOINTS = 120
MAX_PATHS = 2
FCD_WIN_LEN = 30
FCD_WIN_STEP = 5
MODEL_FILTER = None  # e.g. {'Hopf (Grid)', 'Hybrid Hopf', 'Neural SDE'}

SAVE_DIR = PROJECT_ROOT / 'paper_new' / 'images' / 'wholebrain_matrices' / DATASET_TYPE
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = PROJECT_ROOT / '.cache' / 'wholebrain_matrices'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = CACHE_DIR / f'{DATASET_TYPE}_T{N_TIMEPOINTS}_P{MAX_PATHS}.pt'

FORCE_RECOMPUTE = False  # flip to True to ignore the cache

print(f'Dataset     : {DATASET_TYPE}')
print(f'Device      : {DEVICE}')
print(f'Cache file  : {CACHE_PATH}')
print(f'Save figures: {SAVE_DIR}')

## Build or load cached predictions and metrics

In [ ]:
from src.metrics import compute_static_fc, fisher_batch_average
from src.metrics.dynamics_metrics import fcd_matrix, phfcd_matrix


def fc_average(ts):
    return fisher_batch_average(compute_static_fc(ts)).detach().cpu().numpy()


def upper_triangle_corr(a, b):
    idx = np.triu_indices_from(a, k=1)
    return float(np.corrcoef(a[idx], b[idx])[0, 1])


def upper_triangle_mse(a, b):
    idx = np.triu_indices_from(a, k=1)
    return float(np.mean((a[idx] - b[idx]) ** 2))


def subject_real_and_phase(ts, subject_index=0):
    s = ts[subject_index]
    return s.real.T, torch.angle(s).T


def _to_numpy(x):
    return None if x is None else x.detach().cpu().numpy()


def compute_everything():
    from src.dataset import create_data_loaders, load_dataset
    from src.models import load_model_from_checkpoint
    from src.training import HopfConfig

    cfg = HopfConfig()
    cfg.dataset_type = DATASET_TYPE
    cfg.data_path = DATA_PATH
    cfg.use_wandb = False
    cfg.dt_min = 0.05

    dataset = load_dataset(cfg, DEVICE)
    window_size = min(N_TIMEPOINTS, dataset.n_timepoints // 2)

    _, _, _, test_intra_loader = create_data_loaders(
        dataset=dataset,
        window_size=window_size,
        batch_size=cfg.batch_size,
        n_windows_per_epoch=cfg.n_windows_per_epoch,
        train_ratio=cfg.train_ratio,
        val_ratio=cfg.val_ratio,
        seed=cfg.seed,
        device=DEVICE,
    )

    batch = next(iter(test_intra_loader))
    if len(batch) == 4:
        ts_target, _, _patient_ids, control = batch
    elif len(batch) == 3:
        ts_target, _, _patient_ids = batch
        control = None
    else:
        raise ValueError(f'Unexpected batch structure: {len(batch)} items')

    ts_target = ts_target[:MAX_PATHS, :, :N_TIMEPOINTS]
    if control is not None:
        control = control[:MAX_PATHS]
    initial_state = ts_target[:, :, 0]

    _MODEL_TAGS = [
        ('gnn_hopf', 'GNN Hopf'),
        ('hybrid_neural', 'Hybrid+Neural'),
        ('hybrid_hopf', 'Hybrid Hopf'),
        ('nsde', 'Neural SDE'),
        ('hopf', 'Hopf'),
    ]

    def short_label(stem):
        lower = stem.lower()
        for tag, label in _MODEL_TAGS:
            if tag in lower:
                return label + (' (Grid)' if 'grid' in lower else '')
        return stem

    models = {}
    for ckpt_path in sorted(Path(CHECKPOINT_DIR).glob(f'*{DATASET_TYPE}*.pt')):
        model, model_type, _ = load_model_from_checkpoint(str(ckpt_path), device=DEVICE)
        label = short_label(ckpt_path.stem)
        if model.n_rois != dataset.n_rois:
            print(f'Skipping {ckpt_path.name}: ROI mismatch ({model.n_rois} != {dataset.n_rois})')
            continue
        if MODEL_FILTER is not None and label not in MODEL_FILTER:
            continue
        models[label] = (model, model_type)
        print(f'Loaded {label:16s} ({model_type})')

    if not models:
        raise RuntimeError('No compatible checkpoints were loaded.')

    def forward_kwargs_for(model):
        kwargs = {'n_steps': N_TIMEPOINTS}
        if control is not None and getattr(model, 'n_control_dims', 0) > 0 and control.shape[-1] > 0:
            kwargs['control'] = control
        return kwargs

    ts_preds = {}
    for name, (model, _mt) in models.items():
        with torch.no_grad():
            ts_preds[name] = model.forward(initial_state=initial_state, **forward_kwargs_for(model)).cpu()
        print(f'{name:16s} -> {tuple(ts_preds[name].shape)}')

    fc_target = fc_average(ts_target)
    target_real, target_phase = subject_real_and_phase(ts_target, subject_index=0)
    fcd_target = _to_numpy(fcd_matrix(target_real, FCD_WIN_LEN, FCD_WIN_STEP))
    phfcd_target = _to_numpy(phfcd_matrix(target_phase))

    fc_by_model, fcd_by_model, phfcd_by_model = {}, {}, {}
    summary_rows = []
    for name, ts_pred in ts_preds.items():
        fc_pred = fc_average(ts_pred)
        pred_real, pred_phase = subject_real_and_phase(ts_pred, subject_index=0)
        fcd_pred = _to_numpy(fcd_matrix(pred_real, FCD_WIN_LEN, FCD_WIN_STEP))
        phfcd_pred = _to_numpy(phfcd_matrix(pred_phase))

        fc_by_model[name] = fc_pred
        fcd_by_model[name] = fcd_pred
        phfcd_by_model[name] = phfcd_pred

        row = {
            'name': name,
            'fc_corr': upper_triangle_corr(fc_pred, fc_target),
            'fc_mse': upper_triangle_mse(fc_pred, fc_target),
            'fcd_corr': float('nan') if (fcd_target is None or fcd_pred is None) else upper_triangle_corr(fcd_pred, fcd_target),
            'phfcd_corr': float('nan') if (phfcd_target is None or phfcd_pred is None) else upper_triangle_corr(phfcd_pred, phfcd_target),
        }
        summary_rows.append(row)

    summary_rows = sorted(summary_rows, key=lambda r: r['fc_corr'], reverse=True)

    return {
        'ts_target': ts_target.detach().cpu(),
        'ts_preds': {k: v.detach().cpu() for k, v in ts_preds.items()},
        'fc_target': fc_target,
        'fc_by_model': fc_by_model,
        'fcd_target': fcd_target,
        'fcd_by_model': fcd_by_model,
        'phfcd_target': phfcd_target,
        'phfcd_by_model': phfcd_by_model,
        'summary_rows': summary_rows,
        'config': {
            'DATASET_TYPE': DATASET_TYPE,
            'N_TIMEPOINTS': N_TIMEPOINTS,
            'MAX_PATHS': MAX_PATHS,
            'FCD_WIN_LEN': FCD_WIN_LEN,
            'FCD_WIN_STEP': FCD_WIN_STEP,
        },
    }


if CACHE_PATH.exists() and not FORCE_RECOMPUTE:
    print(f'Loading cached results from {CACHE_PATH}')
    results = torch.load(CACHE_PATH, weights_only=False)
else:
    print('Computing results (this runs each model once)...')
    results = compute_everything()
    torch.save(results, CACHE_PATH)
    print(f'Saved cache to {CACHE_PATH}')

fc_target = results['fc_target']
fcd_target = results['fcd_target']
phfcd_target = results['phfcd_target']
fc_by_model = results['fc_by_model']
fcd_by_model = results['fcd_by_model']
phfcd_by_model = results['phfcd_by_model']
summary_rows = results['summary_rows']

for row in summary_rows:
    print(
        f"{row['name']:16s}  FC r={row['fc_corr']:.3f}  "
        f"FC mse={row['fc_mse']:.4f}  FCD r={row['fcd_corr']:.3f}  phFCD r={row['phfcd_corr']:.3f}"
    )

> Multi-trial entropy, Lempel–Ziv–Welch complexity, and spectral clustering / ARI analyses live in `wholebrain_complexity_clustering.ipynb`.

## Plotting helper (no ticks, no labels)

In [ ]:
def _strip(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    for spine in ax.spines.values():
        spine.set_visible(False)


def _save(fig, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches='tight', dpi=200)
    svg = Path(str(path).replace('/images/', '/images_svg/')).with_suffix('.svg')
    svg.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(svg, bbox_inches='tight')
    print(f'Saved {path}')


def plot_matrix_row(items, *, cmap, clim, save_path=None, title=None, cell_size=2.4):
    """Draw one row of square matrices with no ticks, no labels, minimal chrome."""
    n = len(items)
    fig, axes = plt.subplots(1, n, figsize=(cell_size * n, cell_size))
    axes = np.atleast_1d(axes).ravel()
    vmin, vmax = clim
    for ax, (label, matrix) in zip(axes, items):
        if matrix is None:
            ax.axis('off')
            continue
        ax.imshow(np.asarray(matrix), cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')
        ax.set_title(label, fontsize=10)
        _strip(ax)
        ax.set_aspect('equal')
    if title:
        fig.suptitle(title, fontsize=12, y=1.02)
    fig.tight_layout()
    if save_path is not None:
        _save(fig, save_path)
    return fig

## FC per model

In [ ]:
fc_items = [('Target', fc_target)] + [
    (row['name'], fc_by_model[row['name']]) for row in summary_rows
]
plot_matrix_row(
    fc_items,
    cmap='coolwarm',
    clim=(-1.0, 1.0),
    save_path=SAVE_DIR / 'fc_per_model.png',
    title='FC',
)
plt.show()

## FCD per model

In [ ]:
if fcd_target is None:
    print('FCD unavailable for this window size.')
else:
    fcd_items = [('Target', fcd_target)] + [
        (row['name'], fcd_by_model[row['name']]) for row in summary_rows
    ]
    plot_matrix_row(
        fcd_items,
        cmap='viridis',
        clim=(-1.0, 1.0),
        save_path=SAVE_DIR / 'fcd_per_model.png',
        title='FCD',
    )
    plt.show()

## phFCD per model

In [ ]:
if phfcd_target is None:
    print('phFCD unavailable for this selection.')
else:
    phfcd_items = [('Target', phfcd_target)] + [
        (row['name'], phfcd_by_model[row['name']]) for row in summary_rows
    ]
    plot_matrix_row(
        phfcd_items,
        cmap='magma',
        clim=(0.0, 1.0),
        save_path=SAVE_DIR / 'phfcd_per_model.png',
        title='phFCD',
    )
    plt.show()

## Combined grid (FC / FCD / phFCD x models)

In [ ]:
columns = [('Target', None)] + [(row['name'], row['name']) for row in summary_rows]
rows_spec = [
    ('FC', 'coolwarm', (-1.0, 1.0), fc_target, fc_by_model),
    ('FCD', 'viridis', (-1.0, 1.0), fcd_target, fcd_by_model),
    ('phFCD', 'magma', (0.0, 1.0), phfcd_target, phfcd_by_model),
]

n_cols = len(columns)
n_rows = len(rows_spec)
cell = 2.2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(cell * n_cols, cell * n_rows))
axes = np.atleast_2d(axes)

for r, (row_label, cmap, (vmin, vmax), target_mat, by_model) in enumerate(rows_spec):
    for c, (col_label, model_key) in enumerate(columns):
        ax = axes[r, c]
        matrix = target_mat if model_key is None else by_model.get(model_key)
        if matrix is None:
            ax.axis('off')
        else:
            ax.imshow(np.asarray(matrix), cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')
            ax.set_aspect('equal')
        _strip(ax)
        if r == 0:
            ax.set_title(col_label, fontsize=10)
        if c == 0:
            ax.text(-0.12, 0.5, row_label, transform=ax.transAxes,
                    ha='right', va='center', fontsize=11, fontweight='bold')

fig.tight_layout()
_save(fig, SAVE_DIR / 'fc_fcd_phfcd_grid.png')
plt.show()

## Individual per-model SVGs (no title, no labels)

In [ ]:
import re


def _slug(name):
    return re.sub(r'_+', '_', re.sub(r'[^a-z0-9]+', '_', name.lower())).strip('_')


def save_individual_svgs(target_mat, by_model, matrix_name, cmap, clim, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    items = [('target', target_mat)] + [
        (row['name'], by_model[row['name']]) for row in summary_rows
    ]
    for label, matrix in items:
        if matrix is None:
            continue
        fig, ax = plt.subplots(figsize=(2.4, 2.4))
        ax.imshow(np.asarray(matrix), cmap=cmap, vmin=clim[0], vmax=clim[1], interpolation='nearest')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel('')
        ax.set_ylabel('')
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_aspect('equal')
        path = out_dir / f'{matrix_name}_{_slug(label)}.svg'
        fig.savefig(path, bbox_inches='tight', pad_inches=0)
        plt.close(fig)
        print(f'Saved {path}')


INDIVIDUAL_DIR = Path(str(SAVE_DIR).replace('/images/', '/images_svg/')) / 'individual'

save_individual_svgs(fc_target, fc_by_model, 'fc', 'coolwarm', (-1.0, 1.0), INDIVIDUAL_DIR)
if fcd_target is not None:
    save_individual_svgs(fcd_target, fcd_by_model, 'fcd', 'viridis', (-1.0, 1.0), INDIVIDUAL_DIR)
if phfcd_target is not None:
    save_individual_svgs(phfcd_target, phfcd_by_model, 'phfcd', 'magma', (0.0, 1.0), INDIVIDUAL_DIR)